# NEXORA Phase 3 — Data Mining Analysis

Exploratory companion to the production pipelines in `mining/`. This notebook is **not** required to run for the production pipeline (`python -m mining.customer_segmentation`, `python -m mining.association_rules`, `python -m mining.anomaly_detection`, `python -m etl.validate_mining`) — those scripts are fully self-contained. This notebook exists to show the *reasoning* behind their choices: feature selection, K evaluation, threshold tuning, and result interpretation.

Requires the same Snowflake credentials as the rest of the project (`.env`, see `.env.example`) to actually execute; every cell mirrors code that already ran successfully against the live warehouse when the production pipelines were built (see `docs/data_mining.md` for the exact numbers observed).

1. Customer feature exploration
2. K evaluation
3. Silhouette analysis
4. Cluster profile
5. Segment visualization
6. Service basket preparation
7. Association rules
8. Anomaly distribution
9. Business interpretation

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from mining.common import fetch_dataframe
from mining.customer_segmentation import FEATURE_COLUMNS, WINSORIZE_COLUMNS, K_CANDIDATES, prepare_features, load_customer_features, name_segments

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Customer feature exploration

Pull `ANALYTICS.VW_CUSTOMER_360` and look at the raw distributions of the 11 features selected for clustering before any preprocessing — this is what motivated the null-handling and winsorization choices documented in `docs/data_mining.md`.

In [ ]:
customers_raw = load_customer_features()
print(customers_raw.shape)
customers_raw[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").describe().T

In [ ]:
null_counts = customers_raw[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").isna().sum()
null_counts[null_counts > 0]  # AVERAGE_PAYMENT_DELAY, PROJECT_RISK, AVERAGE_SUPPORT_SATISFACTION -- see docs/data_mining.md for why each is null and how each is imputed

In [ ]:
fig, axes = plt.subplots(1, len(WINSORIZE_COLUMNS), figsize=(15, 4))
for ax, col in zip(axes, WINSORIZE_COLUMNS):
    vals = pd.to_numeric(customers_raw[col], errors="coerce").dropna()
    ax.hist(vals, bins=50)
    ax.set_title(f"{col}\np95={vals.quantile(0.95):,.0f}  max={vals.max():,.0f}")
plt.tight_layout()
plt.show()  # heavy right skew on all three -- justifies the 99th-percentile winsorization

## 2. K evaluation

K-Means fit for every K in 2..8, tracking inertia (elbow) and silhouette score for each — K was **not** assumed to be 5.

In [ ]:
features = prepare_features(customers_raw)
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

k_results = []
for k in K_CANDIDATES:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(scaled)
    sil = silhouette_score(scaled, km.labels_)
    k_results.append({"k": k, "inertia": km.inertia_, "silhouette": sil})
k_eval = pd.DataFrame(k_results)
k_eval

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(k_eval["k"], k_eval["inertia"], marker="o")
ax1.set_xlabel("K"); ax1.set_ylabel("Inertia"); ax1.set_title("Elbow (inertia)")
ax2.plot(k_eval["k"], k_eval["silhouette"], marker="o", color="orange")
ax2.set_xlabel("K"); ax2.set_ylabel("Silhouette score"); ax2.set_title("Silhouette vs K")
plt.tight_layout(); plt.show()
# Silhouette peaks at K=3 and declines monotonically afterward -- K=3 selected on this evidence.

## 3. Silhouette analysis

Per-sample silhouette values for the selected K=3, to check whether any cluster is poorly separated (values near/below 0).

In [ ]:
from sklearn.metrics import silhouette_samples

selected_k = 3
kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init=10).fit(scaled)
labels = kmeans.labels_
sample_silhouette = silhouette_samples(scaled, labels)

for c in range(selected_k):
    vals = sample_silhouette[labels == c]
    print(f"Cluster {c}: n={len(vals)}  mean silhouette={vals.mean():.3f}  %negative={100*(vals<0).mean():.1f}%")

## 4. Cluster profile

Raw (unscaled) feature means per cluster -- this table is what the segment names in `docs/data_mining.md` are derived from.

In [ ]:
features_with_cluster = features.copy()
features_with_cluster["CLUSTER_ID"] = labels
profile = features_with_cluster.groupby("CLUSTER_ID").mean()
profile["customer_count"] = features_with_cluster.groupby("CLUSTER_ID").size()
profile

In [ ]:
centers_scaled = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURE_COLUMNS)
segment_names = name_segments(centers_scaled, features.mean())
segment_names  # rule-based names derived from the cluster centroids above -- see mining/customer_segmentation.py::name_segments

## 5. Segment visualization

The 11-dimensional scaled feature space projected to 2D via PCA, colored by cluster, purely for visual intuition (K-Means itself was fit on the full 11-dimensional space, not this projection).

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(scaled)

plt.figure(figsize=(7, 6))
for c in range(selected_k):
    mask = labels == c
    plt.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.5, label=f"Cluster {c}: {segment_names[c]}")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(); plt.title("Customer segments (PCA projection)"); plt.show()

## 6. Service basket preparation

Build the customer-service transaction baskets from CORE.FACT_DEALS (won only) / FACT_SUBSCRIPTIONS (all statuses) / FACT_PROJECTS (non-cancelled) -- see docs/data_mining.md for the full rationale for each source's inclusion rule.

In [ ]:
from mining.association_rules import BASKET_SQL, MIN_SUPPORT, MIN_CONFIDENCE, MIN_LIFT, build_baskets

baskets = build_baskets()
basket_sizes = pd.Series([len(b) for b in baskets])
print(f"{len(baskets)} non-empty baskets")
basket_sizes.describe()

In [ ]:
basket_sizes.value_counts().sort_index().plot(kind="bar", figsize=(8, 4), title="Basket size distribution")
plt.xlabel("distinct items in basket"); plt.ylabel("number of customers"); plt.show()

## 7. Association rules

FP-Growth over the encoded baskets, filtered to the configured support/confidence/lift thresholds (see docs/data_mining.md for how these were tuned).

In [ ]:
from mining.association_rules import mine_rules

rules, n_baskets = mine_rules(baskets)
print(f"min_support={MIN_SUPPORT} min_confidence={MIN_CONFIDENCE} min_lift={MIN_LIFT}")
print(f"{len(rules)} rules survived")
rules.sort_values("lift", ascending=False).head(15)

## 8. Anomaly distribution

Isolation Forest decision-function score distribution for both entity types, with the contamination threshold marked.

In [ ]:
from mining.anomaly_detection import score_customers, score_projects

customer_anomalies = score_customers()
project_anomalies = score_projects()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, [customer_anomalies, project_anomalies], ["Customer anomaly scores", "Project anomaly scores"]):
    ax.hist(df["ANOMALY_SCORE"], bins=50)
    ax.axvline(df.loc[df["IS_ANOMALY"], "ANOMALY_SCORE"].max(), color="red", linestyle="--", label="anomaly cutoff")
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

print("Customer anomaly rate:", customer_anomalies["IS_ANOMALY"].mean())
print("Project anomaly rate:", project_anomalies["IS_ANOMALY"].mean())

In [ ]:
customer_anomalies[customer_anomalies["IS_ANOMALY"]].sort_values("ANOMALY_SCORE").head(10)[["ENTITY_ID", "ANOMALY_SCORE", "OBSERVED_VALUE", "EXPECTED_CONTEXT"]]

## 9. Business interpretation

**Segmentation.** Three behaviorally distinct groups emerged: a small (6.5%) high-value-but-elevated-risk group carrying most of NEXORA's outstanding balance and project risk; a large (33%) genuinely unhealthy group with the worst usage, support, and satisfaction numbers; and a majority (60%) healthy-but-modest-revenue group. No cluster combined high value with high health at K=3 -- itself a useful finding: NEXORA's biggest accounts are not currently its healthiest ones, which is a natural account-management priority, not something the clustering was asked to conclude but a pattern it surfaced.

**Association rules.** The strongest, most consistent pattern is NEXORA Managed Services and NEXORA CRM Suite customers disproportionately also having Implementation, Platform Upgrade, or Data Migration project engagements -- a legitimate cross-sell and delivery-capacity-planning signal, not a causal claim.

**Anomalies.** The customer-level anomalies are dominated by a small set of very-large accounts with proportionally very large outstanding balances -- a concrete, actionable collections/finance signal. The project-level anomalies surface a small number of projects running dramatically over budget -- a concrete delivery-escalation signal. Both are statistical flags for human review, not automated conclusions.

**What this is not.** No model here predicts a future customer outcome, forecasts revenue, or scores churn probability -- that is explicitly out of scope for Phase 3 and reserved for a later, clearly-separated predictive-ML phase.